# Robustness checks on the selection

This notebook checks whether the two chosen compilations depend on choices that were made by convention in `stimulus_selection.ipynb`, not derived from the data. It changes no files.

**What is checked**

| Check | The choice in the main analysis | Varied here |
|---|---|---|
| A | The three ranking measures count equally (strength : coverage : gap = 1 : 1 : 1) | each measure left out, and each counted double |
| B | A clip at 4 SD is a perfectly clear example of a type; a side reached at 2 SD counts as fully covered | 3 and 5 SD; 1.5 and 2.5 SD |
| C | A rotated dimension is kept if it matches its refits at 0.75 or better | 0.70 and 0.80 |

For every variant, the pool is ranked again and each chosen compilation is looked up among the compilations up to the length it was chosen from (59 s for option A, 58 s for option B; step 7f). A compilation is robust to a choice if it stays first, or close to first, under every variant. Shots are weighted by screen time throughout, as in the main analysis.

**How to run it.** Run all cells from the project folder. The first cell runs the whole main notebook (its output is hidden), which takes some minutes; check C repeats the tag test for any dimension it adds, which takes a few minutes more.

In [1]:
%%capture main_notebook_output
# %% 0. Run the main notebook, so that its pool, settings and choices are available here.
#      Its output is captured, not shown; print(main_notebook_output) shows it.
%run stimulus_selection.ipynb

In [2]:
# %% 1. The compilations checked, and the variants
CHECKED = {"option_A_grenzen02": 59,      # compilation (from 8a) -> length limit it was chosen under (7f)
           "option_B_opvang05":  58}
WEIGHT_VARIANTS = {"main (1 : 1 : 1)":   WEIGHTS,
                   "without strength":   {"strength": 0, "coverage": 1, "gap": 1},
                   "without coverage":   {"strength": 1, "coverage": 0, "gap": 1},
                   "without gap":        {"strength": 1, "coverage": 1, "gap": 0},
                   "strength double":    {"strength": 2, "coverage": 1, "gap": 1},
                   "coverage double":    {"strength": 1, "coverage": 2, "gap": 1},
                   "gap double":         {"strength": 1, "coverage": 1, "gap": 2}}
CEILING_VARIANTS = {"strength ceiling 3 SD": (3.0, CEILING), "strength ceiling 5 SD": (5.0, CEILING),
                    "coverage ceiling 1.5 SD": (STRENGTH_CEILING, 1.5), "coverage ceiling 2.5 SD": (STRENGTH_CEILING, 2.5)}
STABLE_VARIANTS  = [0.70, 0.80]

chosen = {name: frozenset(np.where(np.isin(clip_ids, COMPILATIONS[name]))[0]) for name in CHECKED}
results = []

def standing(variant, ranked_pool):
    """Look up every chosen compilation among the compilations up to its length limit, and report it."""
    for name, limit in CHECKED.items():
        cut   = ranked_pool[ranked_pool["seconds"] <= limit].reset_index(drop=True)
        found = [i for i, c in enumerate(cut["clips"]) if frozenset(c) == chosen[name]]
        rank  = found[0] + 1 if found else None
        first = frozenset(cut["clips"].iloc[0])
        swap  = ("" if first == chosen[name] else
                 "first there differs by " + ", ".join([f"- {clip_ids[i]}" for i in sorted(chosen[name] - first)]
                                                        + [f"+ {clip_ids[i]}" for i in sorted(first - chosen[name])]))
        results.append({"variant": variant, "compilation": name, "rank": rank, "of": len(cut)})
        print(f"  {name:<26} rank {rank} of {len(cut)} up to {limit} s   {swap}")

In [3]:
# %% 2. Check A: how much each ranking measure counts
for label, weights in WEIGHT_VARIANTS.items():
    print(label)
    standing(label, build_pool(combinations, durations, sources, has_type, strength_of, sides, CEILING,
                               shot_clip, distances, shot_w, weights))

main (1 : 1 : 1)
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  option_B_opvang05          rank 1 of 73 up to 58 s   
without strength
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  option_B_opvang05          rank 6 of 73 up to 58 s   first there differs by - democracynow_journey_01, - nosop3_opvang_05, - nosop3_grenzen_03, + nieuwsuur_denhaag_05, + nosop3_grenzen_01, + nosop3_grenzen_02
without coverage
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  option_B_opvang05          rank 1 of 73 up to 58 s   
without gap
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  option_B_opvang05          rank 1 of 73 up to 58 s   
strength double
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  option_B_opvang05          rank 1 of 73 up to 58 s   
coverage double
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  option_B_opvang05          rank 1 of 73 up to 58 s   
gap double
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  op

In [4]:
# %% 3. Check B: the ceilings of strength and coverage
for label, (strength_ceiling, ceiling) in CEILING_VARIANTS.items():
    print(label)
    strength_v = type_strength(sides, has_type, type_sides, STRONG, strength_ceiling)
    standing(label, build_pool(combinations, durations, sources, has_type, strength_v, sides, ceiling,
                               shot_clip, distances, shot_w, WEIGHTS))

strength ceiling 3 SD
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  option_B_opvang05          rank 1 of 73 up to 58 s   
strength ceiling 5 SD
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  option_B_opvang05          rank 1 of 73 up to 58 s   
coverage ceiling 1.5 SD
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  option_B_opvang05          rank 1 of 73 up to 58 s   
coverage ceiling 2.5 SD
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  option_B_opvang05          rank 1 of 73 up to 58 s   


In [5]:
# %% 4. Check C: the stability threshold for keeping a dimension
#      A different threshold adds or removes dimensions. The table of content types is kept as decided
#      in step 6 (a new type would need a name and a decision); the check reports whether an added
#      dimension would give a new type, and ranks the pool again with the strength and coverage
#      measured on the new set of dimensions.
type_side_ids = [side_id[s] for s in np.where(is_type)[0]] + [None] * (n_all - int(is_type.sum()))
for threshold in STABLE_VARIANTS:
    keep_v  = np.where(dim_stability >= threshold)[0]
    added   = [int(d) for d in keep_v if d not in keep]
    removed = [int(d) for d in keep if d not in keep_v]
    print(f"stability at least {threshold:.2f}: {len(keep_v)} dimensions; "
          f"added {[d + 1 for d in added] or 'none'}, removed {[d + 1 for d in removed] or 'none'}")

    # Would an added dimension give a content type? (the same tag test as 6b, on the same refits)
    if added:
        share_v  = tag_shares(embeddings, rotated_all, np.array(added), ROTATION, video_of_shot, clip_names,
                              STRONG, np.random.default_rng(0), N_RESAMPLES, SUBSAMPLE)
        labels   = [(d + 1, "high") for d in added] + [(d + 1, "low") for d in added]
        new      = [f"dim {s[0]} {s[1]} ({k} clips)" for s, k in zip(labels, (share_v > MAJORITY).sum(axis=0))
                    if k >= MIN_TYPE_CLIPS]
        print(f"  new content types the added dimensions would give: {', '.join(new) if new else 'none'}")
    lost = [name for name, t in zip(type_names, type_side_ids) if t and t[0] - 1 in removed]
    print(f"  content types whose dimension is removed: {', '.join(lost) if lost else 'none'}")

    # Strength and coverage on the new set of dimensions; the table of types stays as it is.
    z_v         = standardize(rotated_all[:, keep_v])
    sides_v     = clip_sides(z_v, video_of_shot, clip_names)[pd.Index(clip_names).get_indexer(clip_ids)]
    side_id_v   = [(int(d) + 1, "high") for d in keep_v] + [(int(d) + 1, "low") for d in keep_v]
    type_sides_v = [side_id_v.index(t) if t in side_id_v else None for t in type_side_ids]
    strength_v  = type_strength(sides_v, has_type, type_sides_v, STRONG, STRENGTH_CEILING)
    standing(f"stability {threshold:.2f}", build_pool(combinations, durations, sources, has_type, strength_v,
                                                     sides_v, CEILING, shot_clip, distances, shot_w, WEIGHTS))

stability at least 0.70: 17 dimensions; added [14], removed none
  new content types the added dimensions would give: none
  content types whose dimension is removed: none
  option_A_grenzen02         rank 2 of 134 up to 59 s   first there differs by - nos_kroatie_02, - nosop3_grenzen_02, + nieuwsuur_denhaag_01, + nosop3_opvang_05
  option_B_opvang05          rank 1 of 73 up to 58 s   
stability at least 0.80: 14 dimensions; added none, removed [16, 19]
  content types whose dimension is removed: none
  option_A_grenzen02         rank 1 of 134 up to 59 s   
  option_B_opvang05          rank 1 of 73 up to 58 s   


In [6]:
# %% 5. Summary: in how many variants is each compilation first, and how far down does it go at worst?
summary = pd.DataFrame(results)
for name, rows in summary.groupby("compilation", sort=False):
    print(f"{name}: first in {(rows['rank'] == 1).sum()} of {len(rows)} variants; "
          f"worst rank {rows['rank'].max()} (in: {', '.join(rows.loc[rows['rank'] == rows['rank'].max(), 'variant'])})")

option_A_grenzen02: first in 12 of 13 variants; worst rank 2 (in: stability 0.70)
option_B_opvang05: first in 12 of 13 variants; worst rank 6 (in: without strength)
